# nb_write_audit

Called by ADF `pl_bronze_api_payments_v3` via `act_write_audit` (DatabricksNotebook activity). Always runs — whether the pipeline succeeded or failed.

**Upload this notebook to:** `/Shared/ev_intelligence/bronze/nb_write_audit`

Appends one row to `dbw_ev_intelligence_dev.default.pipeline_audit` (managed Delta table inside the `default` schema). Creates the table on first run if it does not exist.

**Audit table schema:**

| Column | Type | Description |
|---|---|---|
| `pipeline_name` | STRING | ADF pipeline name |
| `load_type` | STRING | `full` or `incremental` |
| `watermark_value` | STRING | The `updated_after` value used in this run's API calls |
| `ingestion_date` | STRING | `yyyy-MM-dd` Bronze partition written |
| `total_pages` | INT | Total pages fetched this run |
| `status` | STRING | `succeeded` or `failed` |
| `pipeline_run_id` | STRING | ADF RunId GUID — links back to ADF Monitor |
| `run_timestamp` | TIMESTAMP | UTC time this audit row was written |

`nb_get_watermark` reads `MAX(watermark_value)` from this table on the next incremental run.

## Cell 1 — Receive parameters from ADF

All values are passed as strings from ADF `baseParameters`. `total_pages` is cast to `int` before writing.

In [ ]:
dbutils.widgets.text("pipeline_name",   "pl_bronze_api_payments_v3")
dbutils.widgets.text("load_type",       "incremental")
dbutils.widgets.text("watermark_value", "")
dbutils.widgets.text("ingestion_date",  "")
dbutils.widgets.text("total_pages",     "0")
dbutils.widgets.text("status",          "succeeded")
dbutils.widgets.text("pipeline_run_id", "")

PIPELINE_NAME   = dbutils.widgets.get("pipeline_name")
LOAD_TYPE       = dbutils.widgets.get("load_type")
WATERMARK_VALUE = dbutils.widgets.get("watermark_value")
INGESTION_DATE  = dbutils.widgets.get("ingestion_date")
TOTAL_PAGES     = int(dbutils.widgets.get("total_pages"))
STATUS          = dbutils.widgets.get("status")
PIPELINE_RUN_ID = dbutils.widgets.get("pipeline_run_id")

AUDIT_TABLE     = "dbw_ev_intelligence_dev.default.pipeline_audit"

print(f"Pipeline        : {PIPELINE_NAME}")
print(f"Load type       : {LOAD_TYPE}")
print(f"Watermark used  : {WATERMARK_VALUE}")
print(f"Ingestion date  : {INGESTION_DATE}")
print(f"Total pages     : {TOTAL_PAGES}")
print(f"Status          : {STATUS}")
print(f"Run ID          : {PIPELINE_RUN_ID}")

## Cell 2 — Create audit table if not exists

Creates `pipeline_audit` as a managed Delta table in `dbw_ev_intelligence_dev.default`.

- `IF NOT EXISTS` — idempotent, safe to run on every pipeline execution
- Managed Delta table — Databricks controls the storage location inside the metastore
- `watermark_value` is a STRING so it stores any ISO 8601 timestamp without type issues

In [ ]:
spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {AUDIT_TABLE} (
        pipeline_name    STRING,
        load_type        STRING,
        watermark_value  STRING,
        ingestion_date   STRING,
        total_pages      INT,
        status           STRING,
        pipeline_run_id  STRING,
        run_timestamp    TIMESTAMP
    )
    USING DELTA
    COMMENT 'Pipeline audit log — one row per ADF run. watermark_value drives next incremental load.'
""")

print(f"Audit table ready: {AUDIT_TABLE}")

## Cell 3 — Write audit row

Inserts one row. `current_timestamp()` records when this audit entry was written (UTC).

- `INSERT INTO` always appends — audit history is never overwritten
- Runs for both `succeeded` and `failed` — so failures are visible in the audit log
- `watermark_value` stored here is what `nb_get_watermark` will return on the next incremental run

In [ ]:
spark.sql(f"""
    INSERT INTO {AUDIT_TABLE}
    VALUES (
        '{PIPELINE_NAME}',
        '{LOAD_TYPE}',
        '{WATERMARK_VALUE}',
        '{INGESTION_DATE}',
         {TOTAL_PAGES},
        '{STATUS}',
        '{PIPELINE_RUN_ID}',
        current_timestamp()
    )
""")

print(f"Audit row written — status: {STATUS}, watermark: {WATERMARK_VALUE}")

## Cell 4 — Show recent run history

Displays the last 10 rows for this pipeline so you can see the full run history inline after every execution.

The `watermark_value` column shows exactly what each run used as `updated_after` — confirming the incremental chain is advancing correctly.

In [ ]:
df_history = spark.sql(f"""
    SELECT
        pipeline_name,
        load_type,
        watermark_value,
        ingestion_date,
        total_pages,
        status,
        pipeline_run_id,
        run_timestamp
    FROM   {AUDIT_TABLE}
    WHERE  pipeline_name = '{PIPELINE_NAME}'
    ORDER BY run_timestamp DESC
    LIMIT 10
""")

print(f"Recent runs for {PIPELINE_NAME}:")
display(df_history)